# Week 4: Feature Selection
**Author:** Sakshitha  
**Goal:** Reduce dimensionality by selecting the most relevant features using multiple methods.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
project_path = '/content/drive/MyDrive/ParkinsonsProject/ParkinsonsDetection'
os.chdir(project_path)

import pandas as pd
import numpy as np
import joblib
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif, RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt
from collections import Counter

Mounted at /content/drive


## Load the Balanced, Scaled Data

In [2]:
X_train = np.load('data/processed/X_train.npy')
y_train = np.load('data/processed/y_train.npy')
X_test = np.load('data/processed/X_test.npy')
y_test = np.load('data/processed/y_test.npy')

## Get Original Feature Names
We need the names to interpret results. They are in the same order as the columns after dropping `status` and `name`.

In [4]:
df = pd.read_csv('data/raw/parkinsons.csv')
feature_names = df.drop(['status', 'name'], axis=1).columns.tolist()
print("Total features:", len(feature_names))

Total features: 22


## Method 1: ANOVA F‑value (Filter)
Select top 30 features using F‑score.

In [5]:
selector_f = SelectKBest(score_func=f_classif, k=30)
X_train_f = selector_f.fit_transform(X_train, y_train)
X_test_f = selector_f.transform(X_test)
selected_f = [feature_names[i] for i in selector_f.get_support(indices=True)]
print("ANOVA selected (first 5):", selected_f[:5])

ANOVA selected (first 5): ['MDVP:Fo(Hz)', 'MDVP:Fhi(Hz)', 'MDVP:Flo(Hz)', 'MDVP:Jitter(%)', 'MDVP:Jitter(Abs)']


/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:783: UserWarning: k=30 is greater than n_features=22. All the features will be returned.
  warnings.warn(


## Method 2: Mutual Information (Filter)

In [6]:
selector_mi = SelectKBest(score_func=mutual_info_classif, k=30)
X_train_mi = selector_mi.fit_transform(X_train, y_train)
X_test_mi = selector_mi.transform(X_test)
selected_mi = [feature_names[i] for i in selector_mi.get_support(indices=True)]
print("Mutual info selected (first 5):", selected_mi[:5])

Mutual info selected (first 5): ['MDVP:Fo(Hz)', 'MDVP:Fhi(Hz)', 'MDVP:Flo(Hz)', 'MDVP:Jitter(%)', 'MDVP:Jitter(Abs)']


/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:783: UserWarning: k=30 is greater than n_features=22. All the features will be returned.
  warnings.warn(


## Method 3: Recursive Feature Elimination (Wrapper) with Random Forest

In [7]:
estimator = RandomForestClassifier(random_state=42, n_jobs=-1)
selector_rfe = RFE(estimator, n_features_to_select=30, step=5)
selector_rfe.fit(X_train, y_train)
selected_rfe = [feature_names[i] for i in selector_rfe.get_support(indices=True)]
print("RFE selected (first 5):", selected_rfe[:5])

/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_rfe.py:300: UserWarning: Found n_features_to_select=30 > n_features=22. There will be no feature selection and all features will be kept.
  warnings.warn(


RFE selected (first 5): ['MDVP:Fo(Hz)', 'MDVP:Fhi(Hz)', 'MDVP:Flo(Hz)', 'MDVP:Jitter(%)', 'MDVP:Jitter(Abs)']


## Method 4: Random Forest Feature Importances (Embedded)

In [8]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1][:30]
selected_rf = [feature_names[i] for i in indices]
print("RF importance selected (first 5):", selected_rf[:5])

RF importance selected (first 5): ['PPE', 'MDVP:Fo(Hz)', 'spread1', 'RPDE', 'spread2']


## Combine and Choose Final Set
We'll keep features that appear in at least two of the four methods.

In [9]:
all_selected = selected_f + selected_mi + selected_rfe + selected_rf
counter = Counter(all_selected)
final_selected = [feat for feat, count in counter.items() if count >= 2]

# If more than 30, take top 30 by frequency
if len(final_selected) > 30:
    final_selected = [feat for feat, _ in counter.most_common(30)]

print(f"Final selected features ({len(final_selected)}):")
print(final_selected)

Final selected features (22):
['MDVP:Fo(Hz)', 'MDVP:Fhi(Hz)', 'MDVP:Flo(Hz)', 'MDVP:Jitter(%)', 'MDVP:Jitter(Abs)', 'MDVP:RAP', 'MDVP:PPQ', 'Jitter:DDP', 'MDVP:Shimmer', 'MDVP:Shimmer(dB)', 'Shimmer:APQ3', 'Shimmer:APQ5', 'MDVP:APQ', 'Shimmer:DDA', 'NHR', 'HNR', 'RPDE', 'DFA', 'spread1', 'spread2', 'D2', 'PPE']


## Create Reduced Datasets
Map feature names back to indices.

In [10]:
feature_to_idx = {name: i for i, name in enumerate(feature_names)}
selected_indices = [feature_to_idx[feat] for feat in final_selected]

X_train_fs = X_train[:, selected_indices]
X_test_fs = X_test[:, selected_indices]

print("New shapes:", X_train_fs.shape, X_test_fs.shape)

New shapes: (196, 22) (61, 22)


## Save Results
- Reduced datasets (`.npy`)
- Selected feature names (`.txt`)

In [11]:
np.save('data/processed/X_train_fs.npy', X_train_fs)
np.save('data/processed/X_test_fs.npy', X_test_fs)

with open('data/processed/selected_features.txt', 'w') as f:
    for feat in final_selected:
        f.write(feat + '\n')

print("Feature selection complete. Files saved.")

Feature selection complete. Files saved.


## Done
Proceed to Week 5 (Baseline Models).